# IAN 620 — Qwen Tool Calling Without a System Prompt

A simple **LangChain + LangGraph** tutorial where the language model decides which tools to use from **function docstrings and parameter schemas**. No system prompt, manually written routing instructions, or keyword matching.

Three tools: weather and jobs (clearly labeled dummy results; API calls commented out) and profit (real arithmetic). A question can trigger several tools.

We use the same **Hugging Face Transformers** model-loading approach as the original working LangChain/LangGraph tutorial. **No Ollama.** The small adapter connects Qwen's native tool-call format to LangGraph `ToolNode`.

Run from the top in Google Colab. A T4 GPU runtime is recommended.

## 1. Install libraries

Install the Python packages directly in Colab. No Ollama installation or API key.

In [ ]:
%pip -q install -U langchain langgraph transformers accelerate torch

## 2. Load Qwen from Hugging Face

Same direct-loading method as the original working tutorial. We use Qwen3 1.7B for its chat-template support for tools. A smaller local model may sometimes choose the wrong tool or return malformed parameters.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)
qwen_model.eval()

print("Qwen ready:", MODEL_ID)
print("Device:", next(qwen_model.parameters()).device)

## 3. Describe tools with docstrings

LangChain's `@tool` uses each function's docstring and Python type annotations as its description and schema. Weather and jobs use demo data. Profit is calculated accurately in Python.

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get current weather information for a city.

    Use for requests about current conditions, temperature, rain, or forecasts.
    The city argument should be the location requested by the user.
    """
    # Real API example (requires a weather API provider and key):
    # response = requests.get(WEATHER_API_URL, params={"city": city, "key": API_KEY}, timeout=15)
    # response.raise_for_status()
    # return str(response.json())
    return f"DEMO WEATHER, not live data: {city}: 72 F, partly cloudy."


@tool
def search_jobs(city: str) -> str:
    """Find recent job opportunities in a specified city.

    Use for requests about hiring, openings, employment, or available jobs.
    The city argument should be the location requested by the user.
    """
    # Real API example (requires an authorized jobs API provider):
    # response = requests.get(JOBS_API_URL, params={"location": city}, timeout=15)
    # response.raise_for_status()
    # return str(response.json())
    return (
        f"FICTIONAL JOB EXAMPLES ONLY — not verified openings in {city}:\n"
        "1. Data Analyst — Example Analytics Co.\n"
        "2. Python Developer — Demo Software Inc.\n"
        "3. AI Engineer — Sample Technology Group"
    )


@tool
def calculate_profit(selling_price: float, cost_price: float, quantity: int) -> str:
    """Calculate total profit and profit margin for products.

    Use when asked to compute business profit or earnings from selling price,
    cost per item, and quantity. Pass the actual numeric inputs provided by
    the user; if a required value is missing, ask for it instead of guessing.
    """
    if quantity < 0:
        return "Error: quantity must not be negative."
    revenue = selling_price * quantity
    cost = cost_price * quantity
    profit = revenue - cost
    margin = (profit / revenue * 100) if revenue else 0
    return (
        f"Revenue: ${revenue:,.2f}; Total cost: ${cost:,.2f}; "
        f"Profit: ${profit:,.2f}; Margin: {margin:.1f}%."
    )


tools = [get_weather, search_jobs, calculate_profit]

for item in tools:
    print(item.name, "—", item.description.splitlines()[0])

## 4. Pass the LangChain tool schemas to Qwen

`convert_to_openai_tool()` extracts the schemas that LangChain builds from docstrings. Qwen's chat template receives those schemas directly; **we do not include any system instructions**.

Qwen generates its own tool requests. The parsing code only converts the model's output into LangChain's standard `AIMessage.tool_calls` format; it does not select a tool.

Why not `bind_tools()`? Hugging Face `AutoModelForCausalLM` is not itself a LangChain chat model implementing `bind_tools()`. The native Qwen chat template provides a practical no-Ollama tool-calling path.

In [ ]:
import json
import re
from uuid import uuid4

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

tool_schemas = [convert_to_openai_tool(t) for t in tools]

def to_qwen_message(message):
    """Convert LangChain messages into Qwen's native chat-message format."""
    if isinstance(message, SystemMessage):
        return {"role": "system", "content": message.content}
    if isinstance(message, HumanMessage):
        return {"role": "user", "content": message.content}
    if isinstance(message, ToolMessage):
        return {"role": "tool", "content": str(message.content)}
    if isinstance(message, AIMessage):
        if message.tool_calls:
            calls = [
                {
                    "type": "function",
                    "function": {"name": call["name"], "arguments": call["args"]},
                }
                for call in message.tool_calls
            ]
            return {"role": "assistant", "tool_calls": calls, "content": message.content or ""}
        return {"role": "assistant", "content": str(message.content)}
    raise TypeError(f"Unsupported message: {type(message)}")

def generate_qwen(messages):
    """Ask Qwen to respond, with real tool schemas included in its prompt."""
    chat = [to_qwen_message(m) for m in messages]
    inputs = tokenizer.apply_chat_template(
        chat,
        tools=tool_schemas,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    ).to(qwen_model.device)

    with torch.no_grad():
        output = qwen_model.generate(
            **inputs,
            max_new_tokens=320,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Qwen's actual model-produced tool calls, not keyword-based routing.
    calls = []
    for match in re.finditer(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", generated, re.S):
        try:
            item = json.loads(match.group(1))
            if item.get("name") in {t.name for t in tools}:
                args = item.get("arguments", {})
                if isinstance(args, str):
                    args = json.loads(args)
                if isinstance(args, dict):
                    calls.append({"name": item["name"], "args": args, "id": str(uuid4())})
        except (ValueError, TypeError, KeyError):
            pass

    visible_text = re.sub(r"<tool_call>.*?</tool_call>", "", generated, flags=re.S).strip()
    if "<tool_call>" in generated and not calls:
        visible_text = "I could not read the model's tool request. Please try again."

    return AIMessage(content=visible_text, tool_calls=calls)

preview = generate_qwen([
    HumanMessage(content="What is the temperature in Greensboro?"),
])
print("Model-chosen tool calls:", preview.tool_calls)
print("Model text:", preview.content)

## 5. Build LangGraph with ToolNode

`START → Qwen → ToolNode → Qwen → END`

The only conditional check is whether Qwen actually emitted tool calls. LangGraph executes them and sends the results back. The code does not inspect question wording to select a tool.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def call_model(state: MessagesState):
    response = generate_qwen([
        *state["messages"],
    ])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("llm", call_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition, {"tools": "tools", "__end__": END})
builder.add_edge("tools", "llm")

assistant = builder.compile()
print("Multi-tool LangGraph ready.")

## 6. Inspect actual model-selected tool calls

The printed trace shows the tools selected by Qwen, their arguments, tool results, and the final reply. The model can choose not to call any tools.

In [ ]:
def ask(question: str) -> None:
    print("\nUSER:", question)
    result = assistant.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"recursion_limit": 12},
    )

    for message in result["messages"]:
        if getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print("LLM SELECTED TOOL:", call["name"])
                print("ARGUMENTS:", call["args"])
        if message.type == "tool":
            print("TOOL RESULT:", message.content)
    print("FINAL ANSWER:", result["messages"][-1].content)
    print("-" * 60)

ask("How warm is it in Greensboro?")

## 7. Three example requests

The **jobs and weather results are fictional/sample outputs**, while the profit result uses real arithmetic. Try rephrasing the requests to see whether the model picks tools based on their descriptions.

In [ ]:
ask("Are any companies hiring in Greensboro?")
ask("I buy 10 mugs for 50 dollars each and sell them for 75 each. What is my total profit?")

## 8. One question, multiple tools

This request requires two different sources. Qwen should propose tool calls for both. In the printed trace, look for **both** `search_jobs` and `get_weather`.

Note: Small models can still make incorrect selections. Tool-call accuracy is a model capability, not something the docstring guarantees.

In [ ]:
ask("Are there any jobs in Greensboro, and what is the weather there?")

## 9. Try your own question

There are no manual routing rules. Change the `question` variable and run the cell.

In [ ]:
question = "I sell 4 items for $120 each after buying each for $85. What do I earn?"
ask(question)

## 10. Optional: display the graph

This graph illustrates the model → tools → model loop. It is **not** a hardcoded weather/jobs/profit decision tree.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(assistant.get_graph().draw_mermaid_png()))
except Exception:
    print(assistant.get_graph().draw_mermaid())

## 11. What this notebook shows

- **LangChain `@tool`:** reads the docstrings and type hints and creates tool metadata.
- **Qwen:** reads the metadata in its native chat template and decides whether to call a tool.
- **Adapter:** translates Qwen's returned tool calls into LangChain messages; it **does not route questions**.
- **LangGraph `ToolNode`:** executes calls and returns their outputs to Qwen.
- **No system prompt:** the model sees the user question, chat history, and available tool schemas only.

**Exercise:** Rephrase the weather, jobs, and profit questions. Test a combined request and a general question. Note that docstrings guide selection but do not guarantee accuracy, particularly for a small local model.

The API examples remain commented out; the weather and job outputs are explicitly marked as fictional.

References: [Hugging Face tool use](https://huggingface.co/docs/transformers/main/chat_extras) · [LangGraph ToolNode](https://reference.langchain.com/python/langgraph.prebuilt/tool_node/ToolNode)